# A2.5 · The non-human identity lifecycle

**Function A — AI Architecture, Risks and Mitigations → Controls — Identity and Ingress**  ·  *Security of AI*

Builds on **[A2.4 · Just-in-time authority](https://spbreed.github.io/cyber-commons/lessons/A2.4.html)**.

| | |
|---|---|
| Open-source tooling | SPIFFE/SPIRE, kagent |
| Open-weight models | — |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The hook

Non-human identities already outnumber humans in most estates, and they sit outside joiner-mover-leaver entirely. Nobody ever leaves, so nothing is ever revoked, and last year's proof-of-concept still holds production write.

## 2 · The framework

```
   humans                         non-human identities
   joiner -> mover -> leaver      created -> ... -> ?
      |                              |
   HR system drives it            nothing drives it
   revocation is automatic        nobody ever leaves

   registration . ownership . expiry . revocation  <- the missing four
```

**Mitigates: T9 Identity Spoofing · T13 Rogue Agents · T3 Privilege Compromise.**

Human identities have a lifecycle: someone joins, moves team, leaves, and an HR
event drives the change. Non-human identities have none of that. They are
created by whoever needed one, owned by nobody in particular, and removed never.

They also outnumber humans, often by a large multiple.

A1.10's rogue agent was admitted because the orchestrator had no notion of an
approved agent. The control is a registry, and a registry is only useful if it
carries three fields:

**A named owner.** A person, not a team alias. An identity with no owner cannot
be renewed, questioned or revoked, because nobody is accountable for answering.

**An expiry.** Not for the credential — for the *registration*. It forces a
recurring decision about whether this agent should still exist, which is the
only mechanism that removes the ones nobody uses.

**An admission binding.** The registry entry names the workload identity from
A2.2. Admission then checks the presented identity against the registry rather
than checking a name the caller supplied.

That last point is what makes it a control rather than a spreadsheet. A registry
consulted by name is documentation; a registry consulted by attested identity is
an authorization decision.

> **What this control closes.**
>
> Turns 'which agents are allowed here' from a convention into a check. Closes A1.10 at the door, and makes revoking exactly one agent possible.

## 2 · The control

In [ ]:
NOW = 5000

REGISTRY = {
 "spiffe://corp/pricing-agent": {"owner": "sam@corp", "expires": 9000},
 "spiffe://corp/billing-agent": {"owner": "sam@corp", "expires": 4000},   # lapsed
 "spiffe://corp/legacy-agent":  {"owner": None,       "expires": 9000},   # orphan
}

def admit(presented_identity, now=NOW):
    """Admission checks the ATTESTED identity, not a name the caller supplied."""
    entry = REGISTRY.get(presented_identity)
    if not entry:              return False, "not registered"
    if not entry["owner"]:     return False, "no accountable owner"
    if entry["expires"] < now: return False, "registration lapsed"
    return True, f"owner {entry['owner']}"

PRESENTING = ["spiffe://corp/pricing-agent", "spiffe://corp/billing-agent",
              "spiffe://corp/legacy-agent",  "spiffe://corp/reporting-agent-v2"]

print(f"{'presented identity':36s}{'admitted':10s}why")
admitted = []
for ident in PRESENTING:
    ok, why = admit(ident)
    if ok: admitted.append(ident)
    print(f"{ident:36s}{'yes' if ok else 'NO':10s}{why}")

print(f"\nadmitted {len(admitted)} of {len(PRESENTING)}")
print()
# and revocation is now singular, which A1.7 could not do
REGISTRY["spiffe://corp/pricing-agent"]["expires"] = 0
print("revoke exactly one agent:")
for ident in PRESENTING[:2]:
    ok, why = admit(ident)
    print(f"   {ident:36s}{'admitted' if ok else 'refused'}  ({why})")
print()
print("reporting-agent-v2 is A1.10's rogue: a real process, answering the")
print("protocol, refused because nothing registered it. legacy-agent is the")
print("more common case - registered, running, and owned by nobody.")
assert "spiffe://corp/reporting-agent-v2" not in admitted
assert not admit("spiffe://corp/pricing-agent")[0]

## What you just proved

Four agents present identities and one is admitted: the unregistered one is refused, the lapsed registration is refused, and the orphaned entry with no owner is refused. Revoking a single agent then leaves the others running.

## Your turn

Count your non-human identities and how many have a named human owner. The difference is the set nobody can revoke during an incident, because nobody can be asked whether it is still needed.

---

**Next → [A2.6 · Ingress: marking untrusted content at the door](https://spbreed.github.io/cyber-commons/lessons/A2.6.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A2.5.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A2.5.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*